In [2]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [3]:
sys.path.insert(0, "/lustre/lrspec/users/4301/ABC-SN/code")
from data_degrading import degrade_spectrum
import abcsn_training
import abcsn_config

sys.path.insert(0, "/lustre/lrspec/users/4301/Milligan_project")
from process_data import *


sys.path.insert(0, "/lustre/lrspec/users/4301/snidpy/sourcepy")
from apodize import *
from logwave import Logwave as lw
from logwave import log_rebin

2026-03-18 11:00:59.028042: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-18 11:00:59.085563: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-18 11:01:01.090665: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
abcsn = keras.models.load_model("/lustre/lrspec/users/4301/ABC-SN/abcsn/ABCSN.keras", compile=False)

2026-03-18 11:01:02.334577: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
# used:
# abcsn_config.SN_Stypes_int_to_str replaced with ABC_subtype_id_to_str
# to make the dictionary corresponding to the labels used above
# may just want to change the above to use the same strings as int_to_str function
ABC_subtype_id_to_str = {
    0: "Ia-norm",
    1: "Ia-91T",
    2: "Ia-91bg",
    3: "Iax",
    4: "Ib-norm",
    5: "Ibn",
    6: "IIb",
    7: "Ic-norm",
    8: "Ic-broad",
    9: "IIP",
}

ABC_ID_dict ={"Ia": 0,
          "Iap": 0,
          "Ic": 7,
          "Ib": 4,
          "II": 9, # IIP = type 2 plateau = normal
          "IIb": 6,
          "IIn": 9,
          "SL": None,
          "TDE": None,
          "CaRT": None
          }

Mill_ID_dict ={"Ia": 0,
          "Iap": 0,
          "Ic": 1,
          "Ib": 1,
          "II": 2, # IIP = type 2 plateau = normal
          "IIb": 1,
          "IIn": 2,
          "SL": 3,
          "TDE": 4,
          "CaRT": 4
          }

# five types recorded in Milligan et al.
Mill_types_to_int = {0: "Ia",
                     1: "Ib & Ic",
                     2: "II",
                     3: "SLSN",
                     4: "Non-SN",
                     5: "other"
                     }

# convert ABC types to Mill categories
ABC_to_Mill ={0:0,   # Ia-norm -> Ia
              1:0,   # Ia-91T -> Ia
              2:0,   # Ia-91bg -> Ia
              3:0,   # Iax -> Ia
              4:1,   # Ib-norm -> Ib & Ic
              5:1,   # Ibn -> Ib & Ic
              6:1,   # IIb -> Ib & Ic
              7:1,  # Ic-norm -> Ib & Ic
              8:1,  # Ic-broad -> Ib & Ic
              9:2,  # IIP -> II
            }

In [6]:
folder_list = ["kinney_sample1"]#,
               # "kinney_sample2", "kinney_sample3", "kinney_sample4", "kinney_sample5",
               # "kinney_sample6", "kinney_sample7", "kinney_sample8", "kinney_sample9",
               # "kinney_sample10", "kinney_sample11", "kinney_sample12", "kinney_sample13",
               # "kinney_sample14"]

for folder in tqdm(folder_list):
    filenames = sorted(glob.glob("/lustre/lrspec/users/4301/Milligan_ABC-SN/data/"+ folder +"/*"))
    file_info = [name.split("/")[-1] for name in filenames]
    df_metadata = pd.DataFrame(file_info, columns=["filename"])
    df_metadata[["host", "sn_type", "redshift", "SN_mag", "host_mag"]] = [get_filename_info(info) for info in file_info]
    df_metadata[ "host_mag"] = df_metadata[ "host_mag"].astype(float)
    df_metadata[ "redshift"] = df_metadata[ "redshift"].astype(float)
    df_metadata[ "SN_mag"] = df_metadata[ "SN_mag"].astype(float)

    # set up to extract the X values 
    num_wvl = 139
    dat_size = len(df_metadata)
    plots = False
    
    X_all = np.zeros((dat_size, 1, num_wvl))
    Y_ABC_IDs = np.zeros((dat_size)) # classification as the float classifier
    Y_Mill_IDs = np.zeros((dat_size)) # classification as the float classifier
    
    for i in range(dat_size):
      wvl, X = process_files(filenames[i], 4500, 7000, plot_spectra = plots, verbose=False)
      X_all[i] = X
      # dont save wvl as all are the same
    
      try:
        Y_ABC_IDs[i] = ABC_ID_dict[df_metadata.sn_type[i]]
        Y_Mill_IDs[i] = Mill_ID_dict[df_metadata.sn_type[i]]
    
      except Exception as e:
        print(df_metadata.sn_type[i], i)
        raise e
    df_metadata["ABC_ID"] = Y_ABC_IDs
    df_metadata["Mill_ID"] = Y_Mill_IDs

    # # predict values
    # X = X_all.copy()
    # P = abcsn.predict(X, verbose=0)
    # P_argmax = np.argmax(P, axis=1)
    # df_metadata["Pred_ABC_ID"] = P_argmax
    # df_metadata["Pred_Mill_ID"] = np.array([ABC_to_Mill[i] for i in P_argmax])
    # df_metadata.to_csv(
    #     path_or_buf="/lustre/lrspec/users/4301/Milligan_ABC-SN/data/csvs/" + folder + ".csv", 
    #     index=False, 
    #     lineterminator='\n')


  0%|          | 0/1 [00:00<?, ?it/s]

meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
m

100%|██████████| 1/1 [02:41<00:00, 161.24s/it]


In [7]:
# trimming data (nans are values we do not have in abc-sn rn)
index = ~np.isnan(Y_ABC_IDs)
Y_ABC_IDs = Y_ABC_IDs[index]
X_all = X_all[index]

# train test 
X_tr,  X_te, y_tr, y_te = train_test_split(X_all, Y_ABC_IDs, test_size=0.25, random_state= 7, stratify = Y_ABC_IDs)

# one hot encoded y 
y_tr_ohe = np.zeros((y_tr.shape[0], 10))
for i, y in enumerate(y_tr):
    y_tr_ohe[i][int(y)] = 1
    

In [8]:
abcsn.fit(X_tr, y_tr_ohe)

ValueError: You must call `compile()` before using the model.